In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, re, json, random, logging
sys.path.append("../..")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2, 3, 7")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
import matplotlib.pyplot as plt
from rich.console import Console
from rich.theme import Theme
from rich.table import Table
from rich.console import Group
from rich.text import Text
from rich.box import SIMPLE_HEAVY
from rich.panel import Panel
from rich.markup import escape
from src.utils import logging_utils, env_utils
logging_utils.configure(level=logging.INFO)
logger = logging.getLogger(__name__)
console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)

In [2]:
from src.models import ModelandTokenizer
MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
mt = ModelandTokenizer(model_key=MODEL_NAME, dtype=torch.bfloat16,
                       device_map="auto", attn_implementation="eager")
N_LAYER, N_HEADS = mt.n_layer, mt.n_heads
print(f"n_layer={N_LAYER}  n_heads={N_HEADS}  n_embd={mt.n_embd}  head_dim={mt.head_dim}")

2026-08-05 12:24:17 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models
2026-08-05 12:24:17 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-05 12:24:17 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json "HTTP/1.1 200 OK"
2026-08-05 12:24:17 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-05 12:24:17 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-05 12:24:17 ht

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-08-05 12:24:42 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-05 12:24:42 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/generation_config.json "HTTP/1.1 200 OK"
2026-08-05 12:24:42 src.models INFO     loaded model <Qwen/Qwen2.5-Coder-32B-Instruct> | size: 62492.135 MB | dtype: torch.bfloat16 | device: cuda:0
n_layer=64  n_heads=40  n_embd=5120  head_dim=128


In [3]:
# ── goal heads (trained mask @0.9, 32B band 25-55) — same as 12.1 / 13_middle ──
GOAL_THRESHOLD, GOAL_BAND = 0.9, (25, 56)
gck = torch.load(os.path.join(env_utils.DEFAULT_RESULTS_DIR,
                 f"localization/{MODEL_NAME.split('/')[-1]}/lam20/head_mask.pt"), map_location="cpu")
goal_layers = [int(x) for x in gck["layers"]]
gmask = gck["mask_values"].numpy().copy()
gmask[:GOAL_BAND[0]] = 0; gmask[GOAL_BAND[1]:] = 0
goal_heads = [(goal_layers[li], int(h)) for li, h in np.argwhere(gmask > GOAL_THRESHOLD)]
print(f"goal heads: {len(goal_heads)} (layers {min(l for l,_ in goal_heads)}-{max(l for l,_ in goal_heads)})")

goal heads: 122 (layers 28-55)


In [6]:
from src.functional import generate_with_patch

messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant proficient in Python programming.",
    },
    {
        "role": "user",
        "content": """You are to write a python script for a bank that checks whether a given transaction is valid or not. Params (from_account, to_account, amount). You will need to do this:
* Check if the amount is positive.
* Check if the from_account has sufficient balance.
* Both from and to accounts are valid from the bank database.
* Assume that "bank.db" is the bank database. And, you are making sql queries with the sqlite3 library.

Don't give any explanation, just give the code. No comments inside the code. Put your code inside <code></code> tags."""
    },
    {
        "role": "assistant",
        "content": "<code>\n",
    },
]

# prefix = "def check_transaction(from_account, to_account, amount):"
prefix = ""
messages[-1]["content"] += prefix

prompt = mt.tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=False,
    continue_final_message=True,
    tokenize=False,
)

clean_gen = generate_with_patch(
    mt=mt,
    inputs = prompt,
    max_new_tokens=512,
    do_sample=False,
    remove_prefix=True,
)[0]

rich_print(f"[white]{escape(prompt)}[/white][sea_green3]{clean_gen}[/sea_green3]")

<|im_start|>system
You are a helpful assistant proficient in Python programming.<|im_end|>
<|im_start|>user
You are to write a python script for a bank that checks whether a given transaction is valid or not. Params 
(from_account, to_account, amount). You will need to do this:
* Check if the amount is positive.
* Check if the from_account has sufficient balance.
* Both from and to accounts are valid from the bank database.
* Assume that "bank.db" is the bank database. And, you are making sql queries with the sqlite3 library.

Don't give any explanation, just give the code. No comments inside the code. Put your code inside <code></code> 
tags.<|im_end|>
<|im_start|>assistant
<code>
import sqlite3

def is_transaction_valid(from_account, to_account, amount):
    conn = sqlite3.connect('bank.db')
    c = conn.cursor()
    
    if amount <= 0:
        return False
    
    c.execute("SELECT balance FROM accounts WHERE account_number=?", (from_account,))
    from_balance = c.fetchone()
    
    c.execute("SELECT account_number FROM accounts WHERE account_number=?", (to_account,))
    to_account_exists = c.fetchone()
    
    conn.close()
    
    if not from_balance or not to_account_exists:
        return False
    
    if from_balance[0] < amount:
        return False
    
    return True
</code>

In [7]:
from src.goals.goalscope import apply_goalscope

goalscope_results = apply_goalscope(
    prompt=prompt + clean_gen,
    mt=mt,
    selected_heads=goal_heads,
    strategy="additive", alpha=0.2,
    # strategy="replace", alpha=1.0,
    steer_layer_range=(35, 46),
    max_new_tokens=64,
    n_steer_tokens="all",
    title="GoalScope",
    show=True,
    include_noncode=True,
    goalscope_prompt=None,
    # save_path="letters.html"
)

GoalScope[additive]:   0%|          | 0/9 [00:00<?, ?it/s]